# GraphRAG (2024)
[[paper]](https://arxiv.org/pdf/2404.16130)<br>
GraphRAG = Graph Retrieval-Augmented Generation

__GraphRAG__ — это метод построения систем Retrieval-Augmented Generation, который объединяет извлечение знаний на основе графов (Knowledge Graph) и возможности LLM для решения задач глобального обобщения больших текстовых корпусов.

__Постановка задачи__<br>
Дана большая коллекция неструктурированных текстовых документов $D$. Необходимо ответить на запрос пользователя $Q$, который может носить как локальный характер (конкретный факт), так и глобальный характер (агрегация тем по всему корпусу, например: "Какие основные этические проблемы поднимаются в этих документах?").

__Мотивация__<br>
Классический RAG (2020), основанный на Vector Search и Dense Retrieval, отлично справляется с поиском конкретных фрагментов текста, которые семантически близки к запросу (Local Search). Однако он принципиально не способен решать задачи Global Machine Reading. Если запрос требует понимания всего датасета целиком, стандартный RAG не может эффективно собрать информацию из тысяч разрозненных чанков, так как векторный поиск находит только "соседей", но не видит общую структуру и взаимосвязи между далекими частями текста.

__Существующие подходы__<br>
На момент выхода работы основными альтернативами были:
- Baseline RAG (2020): стандартный подход с использованием векторной базы данных и Top-K retrieval. Проблема: игнорирует структурные связи и не дает глобального контекста.
- Long-context LLM: попытка запихнуть весь корпус документов в контекстное окно современных моделей (типа Gemini 1.5 Pro). Проблема: ограничение по объему (context window всё еще конечно), экспоненциальный рост стоимости и деградация качества (Lost in the Middle).
- Multi-vector RAG: использование иерархических индексов или суммаризаций чанков. Проблема: слабая поддержка многошаговых связей между сущностями.

__Идея__<br>
Вместо того чтобы просто разбивать текст на куски и индексировать их векторами, давайте построим из текста Knowledge Graph, выделим в нем тематические сообщества (Community Detection) и заранее создадим суммаризации (Reports) для каждого такого сообщества. При поиске мы будем обращаться не к исходным текстам, а к этой иерархии отчетов.

__Архитектура__<br>
Система состоит из двух больших блоков: Pipeline индексации и Engine выполнения запросов.
1.  Text Units: исходный текст делится на крупные чанки.
2.  Graph Extraction: LLM проходит по чанкам и извлекает сущности (Entities), их типы и отношения (Relationships).
3.  Graph Enrichment: на основе извлеченных данных строится граф, где узлы — сущности, а ребра — связи.
4.  Community Detection: граф разбивается на иерархические кластеры (сообщества) с помощью алгоритма Leiden. Этот алгоритм группирует узлы, которые сильно связаны друг с другом.
5.  Community Summarization: для каждого обнаруженного сообщества LLM генерирует подробный отчет (Community Report), резюмирующий всю информацию о входящих в него сущностях и связях.

__Алгоритм обучения__<br>
В GraphRAG нет этапа обучения весов нейросети в классическом понимании (Fine-tuning). Обучение заменяется этапом сложной структурированной индексации (Indexing Pipeline):
1.  Экстракция: LLM получает промпт на извлечение триплетов (субъект-предикат-объект) из текста.
2.  Разрешение сущностей (Entity Resolution): устранение дубликатов, когда разные текстовые упоминания относятся к одной сущности.
3.  Генерация отчетов: для каждого уровня иерархии сообществ (от мелких групп до гигантских кластеров) генерируется Summary. Это позволяет подготовить ответы на вопросы разного уровня абстракции.

__Алгоритм инференса__<br>
GraphRAG поддерживает два режима работы в зависимости от типа запроса:
1.  Global Search: используется для вопросов об общей картине. 
    - Запрос рассылается по всем отчетам сообществ определенного уровня иерархии.
    - LLM оценивает релевантность каждого отчета и генерирует промежуточные ответы.
    - Финальная LLM агрегирует эти ответы в один структурированный результат.
2.  Local Search: используется для вопросов о конкретных деталях.
    - Выполняется поиск сущностей, связанных с запросом.
    - Из графа извлекаются соседние узлы, связи и связанные с ними текстовые чанки.
    - Весь этот граф-контекст подается в LLM для генерации ответа.

__Результаты__<br>
Авторы сравнивали GraphRAG с Baseline RAG на наборах данных (например, Podcast transcripts, News articles) по двум метрикам: Comprehensiveness (полнота покрытия аспектов вопроса) и Faithfulness (отсутствие галлюцинаций).
- На глобальных запросах GraphRAG превзошел Baseline RAG в 80% случаев по метрике Comprehensiveness.
- За счет использования Community Reports модель смогла извлекать информацию, которая была разнесена по разным документам и не имела общих ключевых слов, что невозможно для стандартного Dense Retrieval.
- Эффективность системы напрямую коррелирует с уровнем иерархии сообществ: использование среднего уровня иерархии дает оптимальный баланс между детализацией и стоимостью токенов.

## 📝 Критический анализ

# GraphRAG (2024)
---
[[paper]](https://arxiv.org/pdf/2404.16130)<br>
GraphRAG = Graph Retrieval-Augmented Generation

**GraphRAG** — метод для Retrieval-Augmented Generation, объединяющий Knowledge Graph и LLM для глобального обобщения текстов.

## Постановка задачи
Дана коллекция текстов $D$. Нужно ответить на запрос $Q$, который может быть локальным (факт) или глобальным (агрегация тем).

## Мотивация
Классический RAG (2020) хорошо ищет локальные фрагменты, но не справляется с глобальным чтением. Он не видит общую структуру и связи в тексте.

## Существующие подходы
- **Baseline RAG (2020):** использует векторную базу и Top-K retrieval, игнорируя структурные связи.
- **Long-context LLM:** ограничен объемом контекстного окна, что ведет к деградации качества.
- **Multi-vector RAG:** слабая поддержка многошаговых связей.

## Идея
Создание Knowledge Graph из текста, выделение тематических сообществ и создание суммаризаций (Reports) для них. Поиск осуществляется через эту иерархию отчетов.

## Архитектура
Система состоит из Pipeline индексации и Engine выполнения запросов.
1. **Text Units:** текст делится на чанки.
2. **Graph Extraction:** LLM извлекает сущности и отношения.
3. **Graph Enrichment:** строится граф с узлами и связями.
4. **Community Detection:** граф разбивается на кластеры с помощью алгоритма Leiden.
5. **Community Summarization:** для каждого сообщества генерируется отчет.

<img src="img/img.png" width=500>

## Алгоритм обучения
Вместо обучения весов используется сложная индексация:
1. **Экстракция:** LLM извлекает триплеты из текста.
2. **Entity Resolution:** устранение дубликатов сущностей.
3. **Генерация отчетов:** создание Summary для каждого уровня иерархии.

## Алгоритм инференса
GraphRAG поддерживает два режима:
1. **Global Search:** для вопросов об общей картине.
   - Запрос обрабатывается через отчеты сообществ.
   - LLM агрегирует промежуточные ответы.
2. **Local Search:** для конкретных деталей.
   - Поиск сущностей и извлечение соседних узлов.
   - Граф-контекст подается в LLM для ответа.

## Результаты
GraphRAG превосходит Baseline RAG на глобальных запросах в 80% случаев по Comprehensiveness. Использование Community Reports позволяет извлекать информацию из разных документов без общих ключевых слов. Эффективность системы зависит от уровня иерархии сообществ, где средний уровень обеспечивает оптимальный баланс детализации и стоимости.

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Импорт необходимых библиотек
from collections import defaultdict
import networkx as nx
from networkx.algorithms import community
from transformers import pipeline

# Шаг 1: Разделение текста на крупные чанки
documents = [
    "Document 1 text about AI and ethics.",
    "Document 2 text about machine learning and privacy.",
    "Document 3 text about AI, ethics, and privacy."
]

# Шаг 2: Извлечение сущностей и отношений с помощью LLM
# Используем pre-trained модель для извлечения триплетов (субъект-предикат-объект)
triplet_extraction = pipeline("text2text-generation", model="t5-small")

def extract_triplets(text):
    prompt = f"Extract triplets from: {text}"
    result = triplet_extraction(prompt, max_length=50, num_return_sequences=1)
    return result[0]['generated_text']

triplets = [extract_triplets(doc) for doc in documents]

# Шаг 3: Построение графа знаний
G = nx.Graph()

for triplet in triplets:
    # Пример триплета: "AI - related to - ethics"
    entities = triplet.split(", ")
    for entity in entities:
        subject, predicate, obj = entity.split(" - ")
        G.add_node(subject)
        G.add_node(obj)
        G.add_edge(subject, obj, relation=predicate)

# Шаг 4: Обнаружение сообществ с помощью алгоритма Leiden
# Для простоты используем алгоритм Girvan-Newman, который доступен в NetworkX
communities_generator = community.girvan_newman(G)
top_level_communities = next(communities_generator)
communities = sorted(map(sorted, top_level_communities))

# Шаг 5: Генерация отчетов для каждого сообщества
# Используем LLM для генерации суммаризации
summarization = pipeline("summarization", model="facebook/bart-large-cnn")

def generate_community_report(community):
    text = " ".join(community)
    summary = summarization(text, max_length=50, min_length=25, do_sample=False)
    return summary[0]['summary_text']

community_reports = {i: generate_community_report(community) for i, community in enumerate(communities)}

# Шаг 6: Обработка запросов
def global_search(query):
    # Рассылаем запрос по всем отчетам сообществ
    relevant_reports = []
    for report in community_reports.values():
        if query.lower() in report.lower():
            relevant_reports.append(report)
    # Агрегируем ответы
    return " ".join(relevant_reports)

def local_search(query):
    # Поиск сущностей, связанных с запросом
    related_nodes = [node for node in G.nodes if query.lower() in node.lower()]
    # Извлечение соседних узлов и связей
    context = []
    for node in related_nodes:
        neighbors = list(G.neighbors(node))
        context.append((node, neighbors))
    return context

# Пример использования
query_global = "ethics"
query_local = "AI"

global_result = global_search(query_global)
local_result = local_search(query_local)

print("Global Search Result:", global_result)
print("Local Search Result:", local_result)
```

### Комментарии к коду:

1. **Разделение текста на чанки**: Мы начинаем с простого разделения текстов на крупные чанки. В реальной системе это может быть более сложный процесс, учитывающий семантические границы.

2. **Извлечение триплетов**: Используем LLM для извлечения триплетов (субъект-предикат-объект) из текста. Это позволяет нам построить граф знаний.

3. **Построение графа знаний**: На основе извлеченных триплетов строим граф, где узлы — это сущности, а ребра — отношения между ними.

4. **Обнаружение сообществ**: Используем алгоритм обнаружения сообществ для кластеризации графа. В реальной системе можно использовать более сложные алгоритмы, такие как Leiden.

5. **Генерация отчетов**: Для каждого сообщества генерируем суммаризацию с помощью LLM. Это позволяет нам подготовить ответы на запросы разного уровня абстракции.

6. **Обработка запросов**: Реализуем два режима поиска — глобальный и локальный. Глобальный поиск агрегирует информацию из отчетов сообществ, а локальный — извлекает конкретные детали из графа.

Этот код иллюстрирует основные концепции GraphRAG, такие как использование графов знаний и генерация отчетов для сообществ, что позволяет эффективно обрабатывать как локальные, так и глобальные запросы.